In [1]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

from utils.file_utils import get_file_paths_grouped_by_class
from utils.data_loader import load_ms_dataset, load_ms_img_dataset

In [2]:
def load_params_from_yaml(file_path, key=None):
    """
    Load parameters from a YAML file.

    :param file_path: Path to the YAML file.
    :return: Dictionary containing the parameters.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"YAML file {file_path} does not exist.")

    with open(file_path, 'r') as file:
        params = yaml.safe_load(file)

    if not isinstance(params, dict):
        raise ValueError("YAML file must contain a dictionary of parameters.")

    if key:
        if key not in params:
            raise KeyError(f"Key '{key}' not found in the YAML file.")
        return params.get(key)
    else:
        return params

In [30]:
def plot_tsne(X, y, title=None, label_mapping=None, perplexity=30.0):
    # n_pca_components = min(50, X.shape[1])
    # pca = PCA(n_components=n_pca_components, random_state=3407)
    # X_pca = pca.fit_transform(X)
    # tsne = TSNE(n_components=2, perplexity=perplexity, random_state=3407, max_iter=1000)
    # X_tsne = tsne.fit_transform(X_pca)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=3407, max_iter=1000)
    X_tsne = tsne.fit_transform(X)

    df = pd.DataFrame({
    'x': X_tsne[:, 0],
    'y': X_tsne[:, 1],
    'label': y
    })

    reversed_mapping = {v: k for k, v in label_mapping.items()}
    if label_mapping:
        df['Class'] = df['label'].map(reversed_mapping)
    else:
        df['Class'] = df['label'].astype(str)

    plt.figure(figsize=(15, 10))
    sns.set_theme(style='whitegrid', font_scale=1.2)
    sns.scatterplot(
        data=df,
        x='x',
        y='y',
        hue='Class',
        palette='muted',
        s=80,
        alpha=0.6,
        edgecolor='k',
        linewidth=0.5
    )

    plt.title(title, fontsize=40, fontweight='bold', pad=20)
    plt.legend(markerscale=3, fontsize=36, title_fontsize=36, loc='upper right')
    # plt.legend(markerscale=2.5, fontsize=16, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    title = title.replace('\n', '')
    plt.savefig(f"{title}.svg", dpi=300, format='svg')
    # plt.savefig(f"{title}.tiff",dpi=300, pil_kwargs={'compression': 'tiff_lzw'})

In [ ]:
dataset_name = 'CD'
dataset_params = load_params_from_yaml(file_path='../configs/dataset_config.yaml', key=dataset_name)
dataset_dict = {
    'SPNS': f"../datasets/SPNS/PEAK_LIST/Not-Aligned",
    'CD': f"../datasets/CD/PEAK_LIST/Not-Aligned"
}
file_paths_by_class = get_file_paths_grouped_by_class(base_dir=dataset_dict.get(dataset_name), suffix='.csv')
file_paths = [
    {'class_name': class_name, 'file_path': path}
    for class_name, paths in file_paths_by_class.items()
    for path in paths
]
label_mapping = dataset_params.get('label_mapping')
mz_min = dataset_params.get('mz_min')
mz_max = dataset_params.get('mz_max')
bin_size = dataset_params.get('bin_size')
X, y = load_ms_dataset(dataset=file_paths, label_mapping=label_mapping, mz_min=mz_min, mz_max=mz_max, bin_size=bin_size)
title = f"t-SNE Visualization of {dataset_name} Dataset\n(Peak List 1D Representation)"
plot_tsne(X, y, title=title, label_mapping=label_mapping)

In [ ]:
import torch
import torch.nn as nn

dataset_name = 'CD'
patch_strategy = 'DAPS'
num_patches = 64
dataset_params = load_params_from_yaml(file_path='../configs/dataset_config.yaml', key=dataset_name)
dataset_dict = {
    'SPNS': f"../datasets/SPNS",
    'CD': f"../datasets/CD"
}
if patch_strategy == 'GP':
    config_dir_name = f"Entropy_{num_patches}_{patch_strategy}_PATCH_224x224_OVERLAP_0x0_MZ_{dataset_params.get('mz_min')}-{dataset_params.get('mz_max')}_BIN_SIZE_{dataset_params.get('bin_size')}"
    dataset_dir = os.path.join(dataset_dict.get(dataset_name), config_dir_name)
elif patch_strategy == 'DAPS':
    if dataset_name =='SPNS':
        density_threshold = 40
    elif dataset_name == 'CD':
        density_threshold = 45
    else:
        raise ValueError(f"Dataset {dataset_name} not recognized for setting density threshold.")
    config_dir_name = f"Entropy_{num_patches}_{patch_strategy}_PATCH_224x224_WINDOW_100_INT_THR_0.1_DENS_THR_{density_threshold}_MIN_PKS_10_MZ_{dataset_params.get('mz_min')}-{dataset_params.get('mz_max')}_BIN_SIZE_{dataset_params.get('bin_size')}"
    dataset_dir = os.path.join(dataset_dict.get(dataset_name), config_dir_name)
else:
    raise ValueError(f"Patch strategy {patch_strategy} not recognized. Use 'GP' or 'DAPS'.")
file_paths_by_class = get_file_paths_grouped_by_class(base_dir=dataset_dir, suffix='.npz')
file_paths = [
    {'class_name': class_name, 'file_path': path}
    for class_name, paths in file_paths_by_class.items()
    for path in paths
]
label_mapping = dataset_params.get('label_mapping')
X, _, y = load_ms_img_dataset(dataset=file_paths, label_mapping=label_mapping)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled_X = pool(torch.from_numpy(X))
n_samples = X.shape[0]
flattened_X = pooled_X.view(n_samples, -1).numpy()
title = f"t-SNE Visualization of {dataset_name} Dataset \n(Multi-Channel Image Representation Channel={num_patches})"
plot_tsne(flattened_X, y, title=title, label_mapping=label_mapping)